# 🚗 High-Accuracy License Plate Recognition (LPR / ANPR) System

Welcome to the **High-Accuracy License Plate Recognition System**.
This notebook replaces generic, raw OCR pipelines with an **End-to-End 5-Stage System**:

1. **Plate Detection (YOLOv8)**: Detects plate bounding boxes with high precision.
2. **Padded Box Extraction**: Expands bounding box boundaries by a safety margin (8-10%) to prevent character clipping.
3. **Deskewing & Alignment**: Detects tilt angles via contour `minAreaRect` and aligns angled plates horizontally.
4. **Contrast Enhancement**: Applies CLAHE (Contrast Limited Adaptive Histogram Equalization) and adaptive sharpening.
5. **Positional Syntax Correction Engine**: Resolves OCR character confusions (`O` vs `0`, `I` vs `1`, `Z` vs `2`, `S` vs `5`, `B` vs `8`) by enforcing index-based position rules for standard plate formats (e.g., `MH 12 AB 1234`).


## 1. Environment Setup & Dependency Installation

In [ ]:
# Install required packages if running in Colab or fresh environment
!pip install -q ultralytics easyocr opencv-python matplotlib pandas numpy pillow

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import re
import os
import time
from pathlib import Path
from PIL import Image

print("Environment imports successful!")


## 2. Load Core LPR Engine & Modules

In [ ]:
from lpr_engine import (
    LPREngine,
    correct_plate_syntax,
    deskew_plate,
    preprocess_plate_crop,
    crop_with_padding
)

# Initialize LPR Engine
engine = LPREngine()
print("LPR Engine initialized successfully!")


## 3. Positional Syntax Correction Engine Demo

In [ ]:
# Test positional syntax engine on ambiguous OCR inputs:
sample_ocr_outputs = [
    "MH1ZAB1Z34",   # Raw OCR confused '2' as 'Z'
    "KAO5MBS678",   # Raw OCR confused '0' as 'O' and '5' as 'S'
    "DL01CO1234",   # Raw OCR confused 'D' as 'O'
    "UP3ZAB9876",   # Raw OCR confused '2' as 'Z'
    "22BH1234AA"    # Bharat Series
]

print("=== SYNTAX CORRECTION BENCHMARK ===")
for raw in sample_ocr_outputs:
    corrected, is_valid, fmt = correct_plate_syntax(raw)
    print(f"Raw OCR: {raw:<12} -> Corrected: {corrected:<12} | Status: {'VALID' if is_valid else 'INVALID'} ({fmt})")


## 4. End-to-End Image Recognition & Visualization

In [ ]:
def test_lpr_on_image(image_path_or_url, conf=0.35, ocr_engine="easyocr"):
    """Runs full detection, padded crop extraction, deskewing, OCR, and syntax correction."""
    output = engine.process_image(image_path_or_url, conf_thresh=conf, ocr_engine=ocr_engine)
    vis = engine.draw_visualizations(output)
    
    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(f"LPR Results - {len(output['detections'])} Plate(s) Detected", fontsize=14)
    plt.show()
    
    for i, det in enumerate(output["detections"]):
        print(f"--- PLATE #{i+1} ---")
        print(f"BBox: {det['bbox']} | Det Conf: {det['det_conf']:.2f}")
        print(f"Raw OCR Output:       '{det['raw_text']}'")
        print(f"Syntax-Corrected:     '{det['corrected_text']}'")
        print(f"Format Verification:   {det['format_type']} (Valid: {det['is_valid']})\n")

# Example Usage:
# test_lpr_on_image("data/test_car.jpg")


## 5. Quantitative Evaluation Metrics

In [ ]:
def char_level_accuracy(pred_text: str, gt_text: str) -> float:
    """Calculates character-level recognition accuracy between prediction and ground truth."""
    if not gt_text:
        return 0.0
    matches = sum(1 for p, g in zip(pred_text.upper(), gt_text.upper()) if p == g)
    return matches / max(len(pred_text), len(gt_text))

def benchmark_recognition(ground_truth_dict):
    """
    ground_truth_dict: { image_path: ground_truth_plate_str }
    """
    raw_accs = []
    corrected_accs = []
    exact_matches = []
    
    for img_path, gt in ground_truth_dict.items():
        res = engine.process_image(img_path)
        if res["detections"]:
            det = res["detections"][0]
            raw_acc = char_level_accuracy(det["raw_text"], gt)
            corr_acc = char_level_accuracy(det["corrected_text"], gt)
            exact = int(det["corrected_text"] == gt)
        else:
            raw_acc, corr_acc, exact = 0.0, 0.0, 0
            
        raw_accs.append(raw_acc)
        corrected_accs.append(corr_acc)
        exact_matches.append(exact)
        
    print(f"Mean Character Accuracy (Raw OCR):       {np.mean(raw_accs)*100:.2f}%")
    print(f"Mean Character Accuracy (Syntax-Fixed):   {np.mean(corrected_accs)*100:.2f}%")
    print(f"Exact Plate Match Accuracy:               {np.mean(exact_matches)*100:.2f}%")

print("Evaluation benchmark module ready!")


## 6. Real-Time Deployment & Web Dashboard

To run real-time inference on images or live video feeds:
- Launch the Streamlit dashboard: `streamlit run app.py`
- Access the web interface at `http://localhost:8501`
